In [39]:
import numpy as np
import pandas as pd


In [40]:
base_path = "../Dataset/ml-100k/u1.base"
test_path = '../Dataset/ml-100k/u1.test'
train = pd.read_csv(base_path,sep='\t',names=['user_id','item_id','rate','timestamp'])
test = pd.read_csv(test_path,sep = '\t',names = ['user_id','item_id','rate','timestamp'])
print(train.head())


   user_id  item_id  rate  timestamp
0        1        1     5  874965758
1        1        2     3  876893171
2        1        3     4  878542960
3        1        4     3  876893119
4        1        5     3  889751712


In [41]:
train_OCCF = train[train['rate'].isin([4, 5])].copy()
print(f"原始记录数: {len(train)}")
print(train_OCCF.head())


原始记录数: 80000
   user_id  item_id  rate  timestamp
0        1        1     5  874965758
2        1        3     4  878542960
5        1        7     4  875071561
7        1        9     5  878543541
9        1       13     5  875071805


In [42]:
test_OCCF = test[test['rate'].isin([4,5])].copy()
print(f"测试记录数：{len(test)}")
print(test_OCCF.head())


测试记录数：20000
   user_id  item_id  rate  timestamp
0        1        6     5  887431973
2        1       12     5  878542960
3        1       14     5  874965706
5        1       20     4  887431883
6        1       23     4  875072895


In [43]:
def precision_at_k(rec, rel, k):
    return len(set(rec[:k]) & rel) / k if k > 0 else 0.0


def recall_at_k(rec, rel, k):
    return len(set(rec[:k]) & rel) / len(rel) if rel else 0.0


def f1_at_k(rec, rel, k):
    p, r = precision_at_k(rec, rel, k), recall_at_k(rec, rel, k)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


def ndcg_at_k(rec, rel, k):
    rec_k = rec[:k]
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(rec_k) if item in rel)
    ideal_hits = min(len(rel), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0


def one_call_at_k(rec, rel, k):
    return 1.0 if len(set(rec[:k]) & rel) >= 1 else 0.0


def mrr_u(rec, rel):
    for i, item in enumerate(rec):
        if item in rel:
            return 1.0 / (i + 1)
    return 0.0


def ap_u(rec, rel):
    if not rel:
        return 0.0
    hits, score = 0, 0.0
    for i, item in enumerate(rec):
        if item in rel:
            hits += 1
            score += hits / (i + 1)      # i+1 即 1-based 排名
    return score / len(rel)


def arp_u(rec, rel, n_items, n_user_seen):

    if not rel:
        return 0.0
    denom = n_items - n_user_seen
    if denom <= 0:
        return 0.0

    rank = {item: i + 1 for i, item in enumerate(rec)}
    positions = []
    for item in rel:
        if item in rank:
            positions.append(rank[item])
    return float(np.mean(positions) / denom)


def auc_u(pos_scores, neg_scores):

    pos_scores = np.asarray(pos_scores, dtype=np.float64)
    neg_scores = np.asarray(neg_scores, dtype=np.float64)
    n_pos, n_neg = len(pos_scores), len(neg_scores)
    if n_pos == 0 or n_neg == 0:
        return 0.5

    neg_sorted = np.sort(neg_scores)
    # searchsorted(side='left') 返回 neg 中「严格小于 s_i」的个数，
    # 即满足 s_i > s_j 的负样本数（tie 不算，与 δ(>) 一致）。
    strict_wins = np.searchsorted(neg_sorted, pos_scores, side='left').sum()
    return float(strict_wins / (n_pos * n_neg))


# ==================== PopRank 基线 ====================
def build_poprank(train_df, all_items):
    pop = train_df.groupby('item_id').size()
    # 全部物品按流行度降序；训练正样本中没出现过的物品流行度=0，排在最后
    order = sorted(sorted(all_items), key=lambda i: pop.get(i, 0), reverse=True)
    return order, pop.to_dict()


def generate_recommendations(pop_items, test_users, train_df, top_k=None):

    seen = train_df.groupby('user_id')['item_id'].apply(set).to_dict()
    recs = {}
    for u in test_users:
        seen_items = seen.get(u, set())
        rec = [i for i in pop_items if i not in seen_items]
        recs[u] = rec if top_k is None else rec[:top_k]
    return recs


# ==================== 主函数 ====================
def main():
    base_path = "../Dataset/ml-100k/u1.base"
    test_path = '../Dataset/ml-100k/u1.test'

    train = pd.read_csv(base_path, sep='\t', names=['user_id', 'item_id', 'rate', 'timestamp'])
    test = pd.read_csv(test_path, sep='\t', names=['user_id', 'item_id', 'rate', 'timestamp'])

    train_OCCF = train[train['rate'].isin([4, 5])].copy()
    test_OCCF = test[test['rate'].isin([4, 5])].copy()

  
    TRAIN_POS_ITEMS = set(train_OCCF['item_id'])                       # |I^tr| 训练物品集
    POS_ITEMS       = TRAIN_POS_ITEMS | set(test_OCCF['item_id'])     # 全部正样本物品
    N_TRAIN_ITEMS   = len(TRAIN_POS_ITEMS)
    print(f"原始记录数: {len(train)}, 测试记录数: {len(test)}")
    print(f"全部电影数(含 rate<=3): {len(set(train['item_id']) | set(test['item_id']))}")
    print(f"rate>3 正样本物品数 |I^tr| = {N_TRAIN_ITEMS}")

    K = 5

    # ---- PopRank：只在正样本物品上排序 ----
    pop_items, pop_scores = build_poprank(train_OCCF, POS_ITEMS)
    print(f"PopRank 候选物品总数: {len(pop_items)}")

    # ---- ground truth / 已交互集合 ----
    ground_truth = test_OCCF.groupby('user_id')['item_id'].apply(set).to_dict()
    seen_train = train_OCCF.groupby('user_id')['item_id'].apply(set).to_dict()
    test_users = list(ground_truth.keys())
    print(f"测试用户数: {len(test_users)}")

    # ---- 生成完整排序（不再截断到 200）----
    recs = generate_recommendations(pop_items, test_users, train_OCCF, top_k=None)

    # ---- 逐用户计算 ----
    pre_l, rec_l, f1_l, ndcg_l, onecall_l = [], [], [], [], []
    mrr_l, ap_l, arp_l, auc_l = [], [], [], []

    for u in test_users:
        rec = recs[u]
        rel = ground_truth[u]
        seen_u = seen_train.get(u, set())

        pre_l.append(precision_at_k(rec, rel, K))
        rec_l.append(recall_at_k(rec, rel, K))
        f1_l.append(f1_at_k(rec, rel, K))
        ndcg_l.append(ndcg_at_k(rec, rel, K))
        onecall_l.append(one_call_at_k(rec, rel, K))

        mrr_l.append(mrr_u(rec, rel))
        ap_l.append(ap_u(rec, rel))
        # ARP 分母用 |I^tr| - |I_u^tr|（训练正样本物品数，而非全部电影数）
        arp_l.append(arp_u(rec, rel, N_TRAIN_ITEMS, len(seen_u)))

        # AUC 负样本 = 训练正样本物品中，该用户既未训练也未测试的物品
        pos_scores = [pop_scores.get(i, 0.0) for i in rel]
        neg_items = POS_ITEMS - seen_u - rel
        neg_scores = [pop_scores.get(j, 0.0) for j in neg_items]
        auc_l.append(auc_u(pos_scores, neg_scores))

    # ---- 汇总 ----
    result = {
        f"Pre@{K}": np.mean(pre_l),
        f"Rec@{K}": np.mean(rec_l),
        f"F1@{K}": np.mean(f1_l),
        f"NDCG@{K}": np.mean(ndcg_l),
        f"1-call@{K}": np.mean(onecall_l),
        "MRR": np.mean(mrr_l),
        "MAP": np.mean(ap_l),
        "ARP": np.mean(arp_l),
        "AUC": np.mean(auc_l),
    }

    print("\n" + "=" * 30)
    print(f"{'PopRank':>12}")
    print("=" * 30)
    for name, val in result.items():
        print(f"{name:<12} {val:.4f}")
    print("=" * 30)
    print("注：ARP 越小越好（排名百分位），其余指标越大越好。")

    return result


if __name__ == "__main__":
    main()

原始记录数: 80000, 测试记录数: 20000
全部电影数(含 rate<=3): 1682
rate>3 正样本物品数 |I^tr| = 1408
PopRank 候选物品总数: 1447
测试用户数: 456

     PopRank
Pre@5        0.2338
Rec@5        0.0571
F1@5         0.0775
NDCG@5       0.2568
1-call@5     0.5877
MRR          0.4657
MAP          0.1516
ARP          0.1592
AUC          0.8489
注：ARP 越小越好（排名百分位），其余指标越大越好。
